In [17]:
%load_ext autoreload
%autoreload 2
from model.rbm.rbm_two_partite import RBM_TwoPartite
import torch
from hydra.utils import instantiate
from hydra import initialize, compose
import hydra
from omegaconf import OmegaConf
import os
from scripts.run import setup_model, load_model_instance
import wandb
from utils.DWave_Sampling import get_sampler_and_biclique_embedding
from utils.dwave.graphs import run_embedding, analyze_target_side, translate_chain_labels, select_optimal_side, validate_and_repair_chains, build_expanded_embedding
from utils.dwave.physics import convert_energy_to_gray_compact
from utils.dwave.workflows import find_beta_multi_energy, mass_sample_dwave_multi_energy



The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [12]:
SOLVER_NAME = "Advantage2_system1.11" 
hydra.core.global_hydra.GlobalHydra.instance().clear()
initialize(version_base=None, config_path="config")
cfg=compose(config_name="config.yaml")
config = OmegaConf.load(cfg.config_path)
config.gpu_list = cfg.gpu_list
config.load_state = cfg.load_state
self = setup_model(config)
self._model_creator.load_state(config.run_path, self.device)


dummy_data = torch.zeros(1, cfg.rbm.latent_nodes_per_p)
CHECKPOINT_FILE = "/home/leozhu/CaloQuVAE/wandb-outputs/run_2026-02-08_22-42-25_RBM_TwoPartite/training_checkpoint.h5"
rbm = RBM_TwoPartite(cfg, data=dummy_data)

try:
    # Load the latest epoch (epoch=None)
    loaded_epoch = rbm.load_checkpoint(CHECKPOINT_FILE, epoch=None) 
    print(f"Successfully loaded checkpoint from epoch {loaded_epoch}.")
except Exception as e:
    print(f"Error loading checkpoint: {e}")

weights = rbm.params["weight_matrix"]
vbias = rbm.params["vbias"]
print(f"  Weight matrix shape: {weights.shape}")
print(f"  Visible bias mean: {vbias.mean().item():.4f}")
raw_sampler, embedding, qpu_sampler = get_sampler_and_biclique_embedding(rbm.params["vbias"].shape[0]-49, rbm.params["hbias"].shape[0], solver_name=SOLVER_NAME)

wandb.init(tags = [cfg.data.dataset_name], project=cfg.wandb.project, entity=cfg.wandb.entity, config=OmegaConf.to_container(cfg, resolve=True), mode='disabled')

sampler, working_graph, q_used, left_chains, right_chains = run_embedding(rbm.params["hbias"].shape[0], SOLVER_NAME)
best_cond_sets, best_side = select_optimal_side(
    sampler, q_used, left_chains, right_chains, verbose=False
)
n_cond = 49
best_cond_sets = best_cond_sets[:n_cond]

[19:44:07.031] INFO   data.dataManager                                  Loading ATLAS dataset: AtlasCustom2
[19:44:11.950] INFO   data.dataManager                                  Using pre-calculated stratified splits: Tr=399991, Val=49994
[19:44:11.951] INFO   data.dataManager                                  Train: 399991 events, 782 batches
[19:44:11.952] INFO   data.dataManager                                  Val: 49994 events, 49 batches
[19:44:11.953] INFO   data.dataManager                                  Test: 50015 events, 49 batches
[19:44:11.954] INFO   model.modelCreator                                ::Creating Model
[19:44:11.955] INFO   model.autoencoder.autoencoderbase                 Using smoothing_dist: gumbel_no_noise
[19:44:29.229] INFO   model.rbm.zephyr                                  RBM is configured to be fully connected.
[19:44:29.231] INFO   model.rbm.rbm_fulltorch                           RBMTorchFull initialized
[19:44:29.332] INFO   scripts.run      

cuda:4


[19:44:48.150] INFO   model.modelCreator                                Loading state
[19:44:48.234] INFO   model.modelCreator                                Loading weights from file : /fast_scratch_1/caloqvae/lzhu/wandb/run-20260205_232847-4dot02qg/files/ae_separate_best_epoch238.pth
[19:44:48.244] INFO   model.rbm.rbm_two_partite                         Initializing RBM parameters with std: 0.0001
[19:44:48.246] INFO   model.rbm.rbm_two_partite                         Loading parameters from epoch_1999...
[19:44:48.249] INFO   model.rbm.rbm_two_partite                         Loaded persistent chains.
[19:44:48.249] INFO   model.rbm.rbm_two_partite                         Restoring RNG states...
[19:44:48.252] INFO   dwave.cloud.config.models                         Invalid solver JSON, parsing as string identity: 'Advantage2_system1.11'
[19:44:48.264] INFO   dwave.cloud.client.base                           Fetching definition of a solver with name='Advantage2_system1.11'
[19:44:48

Loading weights for module =  _hit_smoothing_dist_mod
Loading weights for module =  _bce_loss
Loading weights for module =  feature_extractor
Loading weights for module =  cond_normalizer
Loading weights for module =  _hit_smoothing
Loading weights for module =  encoder
Loading weights for module =  prior
Loading weights for module =  decoder
Successfully loaded checkpoint from epoch 1999.
  Weight matrix shape: torch.Size([124, 75])
  Visible bias mean: -0.1427
--- Finding Zephyr Embedding for K_75,75 ---


[19:44:49.423] INFO   dwave.cloud.config.models                         Invalid solver JSON, parsing as string identity: 'Advantage2_system1.11'
[19:44:49.436] INFO   dwave.cloud.client.base                           Fetching definition of a solver with name='Advantage2_system1.11'
[19:44:49.460] INFO   dwave.cloud.client.base                           Received solver data for 1 solver(s).
[19:44:49.461] INFO   dwave.cloud.client.base                           Adding solver StructuredSolver(name='Advantage2_system1.11', graph_id='01dceba9f7')


Successfully created embedding for 150 nodes.
--- 1. Running Embedding for K_75,75 on Advantage2_system1.11 ---
Successfully fetched QPU graph with 4582 nodes.
 -> Max chain length: 10
 -> Total qubits used: 1389

--- Analyzing Target Side: Left Chains ---

--- 2. Building 75 Neighbor Sets ---
Total available qubits: 3193
All 76 neighbor sets built.
The 'bottleneck' (min neighbors) is: 62
This is the *absolute upper bound* on the number of nodes.

--- 3. Running Greedy Heuristic for Max Disjoint Hitting Sets ---
------------------------------
Heuristic Result for Left Chains: 51
   (Estimated max number of conditioning nodes)
------------------------------

Physical qubit set sizes for each found node:
617

--- Analyzing Target Side: Right Chains ---

--- 2. Building 75 Neighbor Sets ---
Total available qubits: 3193
All 76 neighbor sets built.
The 'bottleneck' (min neighbors) is: 60
This is the *absolute upper bound* on the number of nodes.

--- 3. Running Greedy Heuristic for Max Disj

In [ ]:
incidence_energies = [1000, 10000, 50000, 100000, 250000]
energy_patterns_dict = {}
for energy in incidence_energies:
    energy_patterns_dict[energy] = convert_energy_to_gray_compact(incidence_energy=energy, engine=self, n_cond=n_cond, num_reads=1024, device=self.device)



tensor([[0., 0., 0.,  ..., 1., 1., 0.],
        [0., 0., 0.,  ..., 1., 1., 0.],
        [0., 0., 0.,  ..., 1., 1., 0.],
        ...,
        [0., 0., 0.,  ..., 1., 1., 0.],
        [0., 0., 0.,  ..., 1., 1., 0.],
        [0., 0., 0.,  ..., 1., 1., 0.]], device='cuda:4')


In [ ]:
optimal_beta, beta_hist, rbm_e_hist, qpu_e_hist = find_beta_multi_energy(
    incidence_energies=incidence_energies,
    energy_patterns_dict=energy_patterns_dict,
    rbm=rbm,
    raw_sampler=raw_sampler,
    left_chains=left_chains,
    right_chains=right_chains,
    conditioning_sets=best_cond_sets,
    hidden_side=best_side,
    lr=0.05
)

test_energy = convert_energy_to_binary(incidence_energy=incidence_energies[0], engine=self, n_cond=n_cond, num_reads=1024, device="cpu")
optimal_beta, beta_hist, rbm_e_hist, qpu_e_hist = find_beta_rigorous(
    rbm=rbm,
    qpu_sampler=qpu_sampler,
    conditioning_sets=best_cond_sets,
    left_chains=left_chains,
    right_chains=right_chains,
    binary_patterns_batch=test_energy,
    hidden_side= best_side,
    num_reads=1024,
    beta_init=3.0,
    lr=0.1,
    num_epochs=50
)


qpu_samples = mass_sample_dwave_multi_energy(
    incidence_energies=incidence_energies,
    energy_patterns_dict=energy_patterns_dict,
    save_dir="/fast_scratch_1/caloqvae/dwave_samples_poster",
    engine=self, rbm=rbm, raw_sampler=raw_sampler, conditioning_sets=best_cond_sets, left_chains=left_chains, right_chains=right_chains,
    beta=optimal_beta, num_srt_batches=8, batch_size=1024, hidden_side=best_side, device=self.device
)


Calculating Global RBM Baseline across 5 energies...
   Energy 1000GeV Baseline: -204.0676
   Energy 10000GeV Baseline: -211.3118
   Energy 50000GeV Baseline: -213.7911
   Energy 100000GeV Baseline: -214.5951
   Energy 250000GeV Baseline: -214.0574
Global Target Mean Energy: -211.5646
------------------------------------------------------------


/usr/local/lib/python3.12/dist-packages/dwave/preprocessing/composites/spin_reversal_transform.py:28: SyntaxWarning: invalid escape sequence '\i'
  """Composite for applying spin reversal transform preprocessing.


SolverFailureError: Problem not accepted because project 4Fq8 has insufficient remaining solver access time.

In [ ]:
plot_beta_optimization(beta_hist, rbm_e_hist, qpu_e_hist, figsize=(10,6))
